In [ ]:
import sys, os, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "flask", "flask-cors", "pyngrok", "openai",
    "sentence-transformers==2.7.0", "timm==0.9.16",
    "faiss-cpu", "requests", "tf-keras", "-q"])
for url in [
    "https://raw.githubusercontent.com/omar-florez/agent-coliseum/bcp/agent_base.py",
    "https://raw.githubusercontent.com/omar-florez/agent-coliseum/bcp/agent_server.py",
    "https://raw.githubusercontent.com/omar-florez/agent-coliseum/bcp/data/latam_facts.jsonl",
]:
    subprocess.check_call(["wget", "-q", "-N", url])
os.environ["AZURE_API_KEY"]  = "YOUR_AZURE_API_KEY_HERE"
os.environ["AZURE_BASE_URL"] = "https://rsgd15-foundry.openai.azure.com/openai/v1/"
os.environ["NGROK_TOKEN"]    = "YOUR_NGROK_TOKEN_HERE"
print(f"Python: {sys.executable}")
print("Setup complete")

In [ ]:
# ============================================================
#  AGENT COLISEUM — BCP Branch — Colab 01: Condor RAG Agent
# ============================================================
#  Strategy: Full agentic agent.
#    - think()   → structured 6-step CoT prompt via Azure Foundry
#    - ask()     → strategic question targeting opponent gaps
#    - answer()  → RAG-augmented answer (TF-IDF + scikit-learn)
#    - move()    → aggressive: seek weakest opponent
#    - memory    → tracks opponent topics and scores across matches
# ============================================================

import os, json, random
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from agent_base import Agent, MatchContext, MatchResult, WorldContext, Position
from agent_server import serve_and_register

# ── Config ────────────────────────────────────────────────────────────────
AZURE_API_KEY  = os.environ.get("AZURE_API_KEY",  "YOUR_AZURE_API_KEY_HERE")
AZURE_BASE_URL = os.environ.get("AZURE_BASE_URL", "https://rsgd15-foundry.openai.azure.com/openai/v1/")
MODEL          = "gpt-5"
ARENA_URL      = "https://agent-coliseum.onrender.com"
NGROK_TOKEN    = os.environ.get("NGROK_TOKEN", "YOUR_NGROK_TOKEN_HERE")

client = OpenAI(base_url=AZURE_BASE_URL, api_key=AZURE_API_KEY)
print(f"Client ready: {client.base_url}")

# ── RAG setup (TF-IDF) ────────────────────────────────────────────────────
# TF-IDF: Term Frequency-Inverse Document Frequency
# Reference: Salton & McGill, 1983 — Introduction to Modern IR
# No GPU, no model download, no API calls — runs anywhere.

def build_rag_index(facts_path="latam_facts.jsonl"):
    facts = []
    with open(facts_path) as f:
        for line in f:
            if line.strip():
                facts.append(json.loads(line))
    texts      = [f["text"] for f in facts]
    vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1, 2),
                                 sublinear_tf=True, min_df=1)
    matrix = vectorizer.fit_transform(texts)
    print(f"RAG index ready: {len(facts)} facts, {matrix.shape[1]} TF-IDF features")
    return vectorizer, matrix, facts

def search_rag(query: str, top_k: int = 3) -> list:
    """Return top-k fact strings most relevant to query via TF-IDF cosine similarity."""
    vec    = rag_vectorizer.transform([query])
    scores = cosine_similarity(vec, rag_matrix).flatten()
    top    = np.argsort(scores)[::-1][:top_k]
    return [rag_facts[i]["text"] for i in top if scores[i] > 0]

rag_vectorizer, rag_matrix, rag_facts = build_rag_index()

# ── Agent ─────────────────────────────────────────────────────────────────

class CondorAgent(Agent):
    """
    Full agentic agent:
    - RAG-augmented answers (TF-IDF, no external dependencies)
    - 6-step chain-of-thought via Azure Foundry GPT-5
    - Opponent memory: tracks topics and scoring patterns
    - Strategic movement: seeks weakest opponent
    """

    name        = "Condor RAG"
    avatar      = "🦅"
    description = "Agente con memoria de oponentes y busqueda semantica de hechos latinoamericanos"

    def __init__(self):
        self._memory = {}

    def on_arena_start(self, ctx: WorldContext) -> None:
        self._memory = {}
        print(f"[{self.name}] Tournament started. {len(ctx.agents)} agents on map.")

    def on_match_start(self, ctx: MatchContext) -> None:
        opp = ctx.opponent_agent_id
        if opp not in self._memory:
            self._memory[opp] = {"name": ctx.opponent_name, "topics_failed": [],
                                  "wins": 0, "losses": 0}
        print(f"[{self.name}] Match vs {ctx.opponent_name} on topic: {ctx.topic}")

    def on_match_end(self, ctx: MatchContext, result: MatchResult) -> None:
        opp = ctx.opponent_agent_id
        won = result.winner_id == ctx.my_agent_id
        if opp in self._memory:
            if won: self._memory[opp]["wins"]   += 1
            else:   self._memory[opp]["losses"] += 1
        print(f"[{self.name}] Match ended. {'WON' if won else 'LOST'}.")

    def on_eliminated(self) -> None:
        print(f"[{self.name}] Eliminated. Final memory: {self._memory}")

    def move(self, ctx: WorldContext) -> Position:
        """Move toward the opponent with the lowest score (easiest target)."""
        active = [a for a in ctx.agents
                  if a.status == "active" and a.agent_id != ctx.my_agent_id]
        if not active:
            dx, dy = random.choice([(0,1),(0,-1),(1,0),(-1,0)])
            return Position(x=max(0, min(ctx.map_width-1,  ctx.my_position.x+dx)),
                            y=max(0, min(ctx.map_height-1, ctx.my_position.y+dy)))
        target = min(active, key=lambda a: a.score)
        dx = 1 if target.position.x > ctx.my_position.x else -1 if target.position.x < ctx.my_position.x else 0
        dy = 1 if target.position.y > ctx.my_position.y else -1 if target.position.y < ctx.my_position.y else 0
        return Position(x=max(0, min(ctx.map_width-1,  ctx.my_position.x+dx)),
                        y=max(0, min(ctx.map_height-1, ctx.my_position.y+dy)))

    def think(self, ctx: MatchContext) -> str:
        """6-step structured CoT. RAG-augmented context injected."""
        rag_hits    = search_rag(ctx.topic + " " + ctx.current_question, top_k=3)
        rag_context = "\n".join(f"- {f}" for f in rag_hits)

        opp_mem = self._memory.get(ctx.opponent_agent_id, {})
        opp_summary = (
            f"Wins: {opp_mem.get('wins',0)}, Losses: {opp_mem.get('losses',0)}, "
            f"Topics struggled: {opp_mem.get('topics_failed', [])}"
            if opp_mem else "No prior history with this opponent."
        )
        history_text = ""
        for t in ctx.history[-3:]:
            history_text += f"\n  Turn {t['turn_number']}: Q={t['question'][:60]} A={t['answer'][:60]} Score={t['score']}"

        prompt = f"""You are playing a Latin America knowledge tournament match.

SITUATION:
- Topic: {ctx.topic}
- Role: {ctx.role}  Turn: {ctx.turn}/{ctx.total_turns}
- My score: {sum(ctx.my_scores)} pts  Opponent: {sum(ctx.opponent_scores)} pts
- Opponent: {ctx.opponent_name}
{f"- Question to answer: {ctx.current_question}" if ctx.role == "answerer" else ""}

OPPONENT PROFILE: {opp_summary}
HISTORY:{history_text if history_text else " (first turn)"}
KNOWLEDGE BASE:
{rag_context}

Think step by step:

# SITUATION  [Chain-of-Thought — Wei et al. 2022, NeurIPS]
# Decompose the current state before acting.
SITUATION: <assess topic, role, turn, score gap>

# OPPONENT  [Theory of Mind — Langner et al. 2023]
# Model what your opponent knows and how they reason.
OPPONENT: <model their knowledge gaps and tendencies>

# GOAL  [ReAct — Yao et al. 2022, ICLR 2023]
# State your concrete objective for this specific turn.
GOAL: <one concrete objective for this turn>

# DRAFT  [Scratchpad — Nye et al. 2021]
# Write your first attempt at the question or answer.
DRAFT: <first attempt>

# CRITIQUE  [Self-Refine — Madaan et al. 2023, NeurIPS]
# Is the draft good enough? What is missing?
CRITIQUE: <evaluate draft quality>

# FINAL  [Reflexion — Shinn et al. 2023]
# Commit to a revised response. Be concise.
FINAL: <final question (1 sentence) or answer (1-2 sentences max)>"""

        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=400,
        )
        return response.choices[0].message.content.strip()

    def ask(self, ctx: MatchContext) -> str:
        """Generate a strategic question. Extracts FINAL line from think()."""
        return self._extract_final(self.think(ctx))

    def answer(self, ctx: MatchContext) -> str:
        """Generate a RAG-augmented answer. Extracts FINAL line from think()."""
        return self._extract_final(self.think(ctx))

    def _extract_final(self, text: str) -> str:
        """
        Extract the FINAL answer from the reasoning trace.
        Robust to GPT formatting: "FINAL: ...", "**FINAL:**", FINAL on own line.
        Falls back to last non-empty non-comment line.
        """
        lines = text.split("\n")
        for i, line in enumerate(lines):
            clean = line.replace("**", "").strip()
            if clean.upper().startswith("FINAL:"):
                rest = clean[6:].strip()
                if rest:
                    return rest
                for j in range(i+1, len(lines)):
                    if lines[j].strip():
                        return lines[j].strip()
            elif clean.upper() == "FINAL":
                for j in range(i+1, len(lines)):
                    if lines[j].strip():
                        return lines[j].strip()
        for line in reversed(lines):
            if line.strip() and not line.strip().startswith("#"):
                return line.strip()
        return text.strip()

# ── Keepalive ─────────────────────────────────────────────────────────────
import threading, requests as _req, time as _time

def _keepalive(arena_url):
    """Ping arena every 60s to keep ngrok tunnel and Colab session alive."""
    while True:
        _time.sleep(60)
        try:
            _req.get(f"{arena_url}/health", timeout=5)
        except:
            pass

threading.Thread(target=_keepalive, args=(ARENA_URL,), daemon=True).start()
print("Keepalive thread started")

# ── Run ───────────────────────────────────────────────────────────────────
agent = CondorAgent()

In [ ]:
# ── DEBUG CELL — run this to inspect raw GPT-5 output ────────────────────
# This helps verify that _extract_final works correctly.
# Run this cell after the agent is defined, before serve_and_register.

from agent_base import MatchContext

test_ctx = MatchContext(
    match_id="test", topic="Latin American geography",
    turn=1, total_turns=3, role="asker",
    history=[], my_agent_id="me", opponent_agent_id="opp",
    opponent_name="Test Opponent", my_scores=[], opponent_scores=[],
)

print("=== RAW think() OUTPUT ===")
raw = agent.think(test_ctx)
print(raw)
print()
print("=== EXTRACTED FINAL ===")
print(repr(agent._extract_final(raw)))
print()

test_ctx_answerer = MatchContext(
    match_id="test", topic="Latin American geography",
    turn=1, total_turns=3, role="answerer",
    history=[], my_agent_id="me", opponent_agent_id="opp",
    opponent_name="Test Opponent", my_scores=[], opponent_scores=[],
    current_question="What is the longest river in South America?",
)

print("=== RAW think() OUTPUT (answerer) ===")
raw2 = agent.think(test_ctx_answerer)
print(raw2)
print()
print("=== EXTRACTED FINAL ===")
print(repr(agent._extract_final(raw2)))

In [ ]:
# ── Run cell — starts the agent server ───────────────────────────────────
from pyngrok import ngrok
ngrok.kill()  # kill any existing tunnels before starting

serve_and_register(
    agent       = agent,
    arena_url   = ARENA_URL,
    port        = 5000,
    ngrok_token = NGROK_TOKEN,
)
# This cell blocks. Agent is now live and registered.
# Wait for the organizer to accept you in the admin panel.